# Análisis Exploratorio de Datos — MetroPT3

Dataset de telemetría de un compresor de aire del metro de Porto.  
Contiene **8 sensores analógicos** y **7 sensores digitales** muestreados cada ~10 segundos durante varios meses de 2020.

**Objetivo:** entender la estructura, distribuciones y propiedades temporales de los datos crudos, *antes de cualquier transformación o etiquetado*.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

sns.set_style('whitegrid')

ANALOG_COLS  = ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs',
                'Oil_temperature', 'Motor_current', 'Caudal_impulses']
BINARY_COLS  = ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS',
                'Pressure_switch', 'Oil_level']

In [ ]:
df = pd.read_csv('../data/raw/MetroPT3.csv', parse_dates=['timestamp'])
df.drop(columns=['Unnamed: 0'], inplace=True)
df = df.sort_values('timestamp').reset_index(drop=True)
df.head()

## 1. Estructura y calidad del dataset

In [ ]:
print(f"Filas × columnas : {df.shape}")
print(f"Rango temporal   : {df['timestamp'].min()}  →  {df['timestamp'].max()}")
print(f"Duración total   : {df['timestamp'].max() - df['timestamp'].min()}")

# Frecuencia de muestreo
deltas = df['timestamp'].diff().dropna()
print(f"\n--- Frecuencia de muestreo ---")
print(f"  Mediana : {deltas.median()}")
print(f"  Mínimo  : {deltas.min()}")
print(f"  Máximo  : {deltas.max()}")

# Gaps significativos (> 1 minuto)
gaps = deltas[deltas > pd.Timedelta('1min')].sort_values(ascending=False)
print(f"\nGaps > 1 min : {len(gaps)}")
if len(gaps):
    print(gaps.head(10).to_string())

# Calidad
print(f"\n--- Calidad ---")
print(f"Valores nulos  :\n{df.isnull().sum().to_string()}")
print(f"\nDuplicados     : {df.duplicated().sum()}")

## 2. Propiedades estadísticas

In [ ]:
print("=== Sensores analógicos ===")
display(df[ANALOG_COLS].describe().T
        .style.background_gradient(cmap='Blues', subset=['mean', 'std', 'min', 'max'])
        .format(precision=4))

In [ ]:
print("=== Sensores binarios ===")
binary_stats = pd.DataFrame({
    'activo (%)' : (df[BINARY_COLS].mean() * 100).round(2),
    'n activo'   : df[BINARY_COLS].sum().astype(int),
    'n inactivo' : (len(df) - df[BINARY_COLS].sum()).astype(int),
})
display(binary_stats.style.background_gradient(cmap='Oranges', subset=['activo (%)']))

## 3. Distribuciones de sensores analógicos

Histogramas con KDE. Las líneas punteadas marcan media (roja) y mediana (naranja).  
Distribuciones bimodales indican al menos dos modos de operación del compresor.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, ANALOG_COLS):
    sns.histplot(df[col], bins=60, kde=True, ax=ax, color='steelblue', alpha=0.7)
    ax.axvline(df[col].mean(),   color='red',    linestyle='--', linewidth=1.2, label='media')
    ax.axvline(df[col].median(), color='orange',  linestyle='--', linewidth=1.2, label='mediana')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')

axes[0].legend(fontsize=8)
plt.suptitle('Distribuciones de sensores analógicos (datos crudos)', fontsize=14, fontweight='bold')
plt.tight_layout()

## 4. Sensores binarios — activación y comportamiento temporal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Izquierda: proporción activa por sensor
proportions = df[BINARY_COLS].mean().sort_values() * 100
axes[0].barh(proportions.index, proportions.values, color='steelblue', edgecolor='white')
axes[0].set_xlabel('% del tiempo activo')
axes[0].set_title('Proporción de activación por sensor', fontweight='bold')
for i, v in enumerate(proportions.values):
    axes[0].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

# Derecha: estado de los sensores en la primera semana (vista tipo Gantt)
week = df[df['timestamp'] < df['timestamp'].min() + pd.Timedelta('7D')].copy()
colors = plt.cm.tab10.colors
for i, col in enumerate(BINARY_COLS):
    on_mask = week[col] == 1
    axes[1].fill_between(week['timestamp'], i + week[col], i,
                         where=on_mask, color=colors[i], alpha=0.8, label=col)

axes[1].set_yticks(np.arange(len(BINARY_COLS)) + 0.5)
axes[1].set_yticklabels(BINARY_COLS)
axes[1].set_title('Estado de sensores binarios — primera semana', fontweight='bold')
axes[1].set_xlabel('Fecha')

plt.tight_layout()

## 5. Series de tiempo completas

Vista de los datos crudos a lo largo de todo el período de registro.

In [ ]:
# Resampleo horario solo para visualización (los datos originales son cada ~10 s)
df_h = df.set_index('timestamp')[ANALOG_COLS].resample('1h').mean().reset_index()

fig, axes = plt.subplots(len(ANALOG_COLS), 1, figsize=(16, 14), sharex=True)

for ax, col in zip(axes, ANALOG_COLS):
    ax.plot(df_h['timestamp'], df_h[col], linewidth=0.7, color='steelblue')
    ax.set_ylabel(col, fontsize=9, fontweight='bold')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Fecha')
plt.suptitle('Sensores analógicos — serie completa (media horaria)', fontsize=13, fontweight='bold')
plt.tight_layout()

## 6. Correlación entre sensores

In [ ]:
corr = df[ANALOG_COLS].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            mask=mask, square=True, linewidths=0.5)
plt.title('Correlación entre sensores analógicos (triángulo inferior)', fontweight='bold')
plt.tight_layout()

## 7. Autocorrelación — memoria temporal de los sensores

El ACF indica cuántos pasos hacia atrás "recuerda" cada sensor.  
Esto define si los modelos necesitarán ventanas temporales (rolling features, LSTM, etc.).

In [ ]:
key_sensors = ['TP2', 'TP3', 'Oil_temperature', 'Motor_current']
LAGS = 200  # ~33 minutos a 10 s por muestra

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, key_sensors):
    plot_acf(df[col].dropna(), lags=LAGS, ax=ax, color='steelblue',
             alpha=0.05, zero=False)
    ax.set_title(f'ACF — {col}', fontweight='bold')
    ax.set_xlabel('Lag (× 10 s)')
    ax.set_ylabel('Autocorrelación')

plt.suptitle('Función de Autocorrelación — sensores clave', fontsize=13, fontweight='bold')
plt.tight_layout()